In [ ]:
!pip install SpeechRecognition pydub matplotlib
!apt-get install -y ffmpeg
!python -m textblob.download_corpora

from google.colab import files
uploaded = files.upload()

In [ ]:
!pip install pydub
import speech_recognition as sr
from pydub import AudioSegment
import re

# Load & transcribe
file_name = list(uploaded.keys())[0]

# Convert the audio file to WAV format
if file_name.lower().endswith(('.m4a', '.mp3', '.ogg', '.flac')): # Check if it's a format we need to convert
    sound = AudioSegment.from_file(file_name, format=file_name.split('.')[-1]) # Use pydub to load
    sound.export("temp_audio.wav", format="wav") # Export as WAV
    file_name = "temp_audio.wav" # Update file_name for SpeechRecognition

recognizer = sr.Recognizer()

with sr.AudioFile(file_name) as source:
    audio_data = recognizer.record(source)
    transcript = recognizer.recognize_google(audio_data)

print(" Transcript:\n", transcript)

# Load audio for timing/volume
audio = AudioSegment.from_wav(file_name)
duration_minutes = len(audio) / (1000 * 60)
word_count = len(transcript.split())
wpm = word_count / duration_minutes
volume = audio.dBFS

In [ ]:
filler_words = ['um', 'uh', 'like', 'you know', 'so']
count = {w: len(re.findall(rf'\b{w}\b', transcript, re.IGNORECASE)) for w in filler_words}
total_fillers = sum(count.values())

In [ ]:
def give_feedback(wpm, volume, filler_count):
    tips = []

    if wpm < 110:
        tips.append(" Try increasing your speaking speed. Practice with tongue twisters.")
    elif wpm > 160:
        tips.append(" Slow down a bit to sound more controlled and clear.")
    else:
        tips.append(" Great pacing!")

    if volume < -30:
        tips.append(" Speak louder with more energy.")
    elif volume > -10:
        tips.append(" Volume is a bit high — try balancing projection.")
    else:
        tips.append(" Volume is solid!")

    if filler_count > 5:
        tips.append(" Work on reducing filler words. Record yourself and pause instead of saying 'um'.")

    return tips

feedback = give_feedback(wpm, volume, total_fillers)

In [ ]:
import matplotlib.pyplot as plt

labels = ['Words Per Minute', 'Volume (dBFS)', 'Filler Words']
values = [wpm, volume, total_fillers]

plt.figure(figsize=(8,5))
plt.bar(labels, values, color=['skyblue', 'lightgreen', 'salmon'])
plt.title(" Speech Performance Overview")
plt.ylabel("Value")
plt.show()

In [ ]:
import random
quotes = [
    "Believe in your voice. Someone out there needs to hear it.",
    "Confidence comes from progress, not perfection.",
    "Speak your truth—even if your voice shakes.",
]

print("\n FULL SPEECH PERFORMANCE REPORT")
print(f"- Words Per Minute: {wpm:.2f}")
print(f"- Volume (dBFS): {volume:.2f}")
print(f"- Filler Words Used: {total_fillers}")
print(f"- Pitch Std Dev (Emotion): {pitch_std:.2f}")
print(f"- Syllables/Min (Enunciation): {syllables_per_minute:.2f}")
print(f"- Confidence Score: {confidence_score:.2f}")
print("\n Confidence Quote of the Day:", random.choice(quotes))

# 🎤 Vocal Coach Insight
print("\n Coaching Tips:")
for tip in feedback + emotion_feedback:
    print("•", tip)

# Enunciation
print("\n Enunciation Feedback:", enunciation)

# Emotion Insight
if pitch_std < 25:
    print(" Emotion Feedback: Your tone is a little flat. Add more highs and lows to sound expressive.")
else:
    print(" Emotion Feedback: Great pitch variety! You sound emotionally engaged.")

#  Practice Ideas
print("\n Targeted Practice Ideas:")
if pitch_std < 25:
    print("- Practice reading a dramatic monologue. Over-act it to train vocal emotion!")
if syllables_per_minute > 300:
    print("- Read a poem slowly and clearly, pausing after each line.")
if syllables_per_minute < 180:
    print("- Record a high-energy topic like a sports recap or a favorite movie review.")

In [ ]:
from datetime import datetime

date = datetime.now().strftime("%Y-%m-%d")
journal_entry = input("How did you feel about your speech today? Any goals for next time?\n")

with open("confidence_journal.txt", "a") as f:
    f.write(f"\n[{date}]\n")
    f.write(f"Transcript Excerpt: {transcript[:100]}...\n")
    f.write(f"Stats - WPM: {wpm:.1f}, Volume: {volume:.1f}, Fillers: {total_fillers}\n")
    f.write(f"Reflection: {journal_entry}\n")
    f.write("Coach Feedback: " + "; ".join(feedback) + "\n")

In [ ]:
def practice_exercises(wpm, volume, filler_count):
    print("\n Personalized Practice Suggestions:")

    if wpm < 110:
        print("- Read aloud a news article and try to speed up naturally.")
    if filler_count > 3:
        print("- Record 1-min speeches and pause when unsure instead of using 'um'.")
    if volume < -30:
        print("- Stand and project your voice as if talking to someone 10 feet away.")

practice_exercises(wpm, volume, total_fillers)

In [ ]:
import random

quotes = [
    "Confidence is a muscle — train it daily.",
    "You’re not just speaking — you’re owning your truth.",
    "Every word is a step toward power.",
    "Your story is worth hearing. Speak it boldly."
    "Believe in your voice!",
    "Progress, not perfection.",
    "Confidence grows with every word.",
    "Your story matters. Speak it.",
    "Believe you can and you're halfway there.",
    "Never bend your head. Always hold it high. Look the world straight in the eye.",
    "Believe in your infinite potential. Your only limitations are those you set upon yourself.",
    "All the great speakers were bad speakers at first.",
    "There is no limit to what you can do."
]

print("\n Daily Confidence Quote:")
print(random.choice(quotes))
print("Challenge: Record a 1-minute story about your day.")

In [ ]:
import numpy as np
import wave
import contextlib
import os

# Function to get pitch (simplified for beginners)
def estimate_pitch(wav_file):
    import librosa
    y, sr = librosa.load(wav_file)
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
    pitch_values = pitches[magnitudes > np.median(magnitudes)]
    pitch_mean = np.mean(pitch_values)
    pitch_std = np.std(pitch_values)
    return pitch_mean, pitch_std

pitch_mean, pitch_std = estimate_pitch(file_name)

In [ ]:
syllables = sum([len(re.findall(r'[aeiouy]+', word.lower())) for word in transcript.split()])
syllables_per_minute = syllables / duration_minutes

if syllables_per_minute > 300:
    enunciation = " You're speaking too fast for clear enunciation."
elif syllables_per_minute < 180:
    enunciation = " You can add a little more energy and pace."
else:
    enunciation = " Your enunciation rate is clear and strong!"

In [ ]:
def coach_emotion(pitch_std, syllables_per_minute):
    advice = []
    if pitch_std < 25:
        advice.append(" Try adding more vocal variety to express emotion. Emphasize key words with tone shifts.")
    else:
        advice.append(" Good pitch variety! You're expressive.")

    if syllables_per_minute > 300:
        advice.append(" You're speaking very quickly. Slow down to enunciate more clearly.")
    elif syllables_per_minute < 180:
        advice.append(" Increase your energy a bit for better impact.")
    else:
        advice.append(" Your pacing supports clarity well.")

    return advice

emotion_feedback = coach_emotion(pitch_std, syllables_per_minute)

print("\n Vocal Coach Feedback:")
for e in emotion_feedback:
    print("-", e)

In [ ]:
print("\n Targeted Practice Ideas:")

if pitch_std < 25:
    print("- Practice reading a dramatic monologue. Over-act it to train vocal emotion!")
if syllables_per_minute > 300:
    print("- Read a poem slowly and clearly, pausing after each line.")
if syllables_per_minute < 180:
    print("- Record a high-energy topic like a sports recap or a favorite movie review.")

In [ ]:
confidence_score = (100 - total_fillers*2 + pitch_std + wpm/2) / 2

In [ ]:
# Step 1: Install dependencies
!pip install pydub speechrecognition librosa ipywidgets --quiet

# Step 2: Import everything
import os
import librosa
import numpy as np
import speech_recognition as sr
from pydub import AudioSegment
from IPython.display import display, Audio, Markdown
import ipywidgets as widgets
import random
import tempfile

In [ ]:
# Confidence quotes and goals
quotes = [
    "Confidence comes from preparation. Practice makes progress!",
    "Speak like you mean it – your voice matters.",
    "Don’t aim for perfection, aim to be heard.",
    "Even the best speakers use filler words. Keep growing!"
]

goals = [
    " Record 1 short speech today!",
    " Practice removing 3 filler words in your next speech.",
    " Speak clearly for 30 seconds without stopping.",
    " Focus on volume control in your next practice round."
]

def display_feedback(text, speed, volume, filler_count, emotion_score, clarity_score):
    display(Markdown(f"## Feedback Summary"))

    if speed < 100:
        print(" Your speaking pace is a bit slow. Try increasing your energy!")
    elif speed > 170:
        print(" You're speaking quite fast! Try pausing to let ideas land.")
    else:
        print(" Great pace! You're speaking at a confident speed.")

    if volume < 0.05:
        print(" Your volume is low. Speak up to sound more confident.")
    elif volume > 0.3:
        print(" Your volume is a bit loud. Try to speak steadily.")
    else:
        print(" Volume is good! Clear and confident.")

    if filler_count > 5:
        print(f" You used {filler_count} filler words. Try to pause instead of saying 'um' or 'like'.")
    else:
        print(f" Minimal filler words – well done!")

    if emotion_score < 0.2:
        print(" Try adding more emotion! Your speech feels flat.")
    else:
        print(" Good energy and tone variation – keep it up!")

    if clarity_score < 0.3:
        print(" Work on enunciation. Try reading aloud with clear articulation.")
    else:
        print(" Your enunciation is pretty clear!")

    print("\n Confidence Quote: ", random.choice(quotes))
    print(" Daily Goal: ", random.choice(goals))

In [ ]:
def analyze_speech(file_path):
    y, sr_lib = librosa.load(file_path)

    # Speed: estimate words per minute
    duration_sec = librosa.get_duration(y=y, sr=sr_lib)
    recognizer = sr.Recognizer()
    with sr.AudioFile(file_path) as source:
        audio = recognizer.record(source)
        try:
            text = recognizer.recognize_google(audio)
        except:
            text = ""

    word_count = len(text.split())
    wpm = (word_count / duration_sec) * 60

    # Filler words
    fillers = ['um', 'uh', 'like', 'you know', 'so']
    filler_count = sum(text.lower().count(filler) for filler in fillers)

    # Volume (RMS)
    volume = np.mean(librosa.feature.rms(y=y))

    # Basic emotion: pitch variation
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr_lib)
    emotion_score = np.std(pitches[pitches > 0]) / np.mean(pitches[pitches > 0]) if np.mean(pitches[pitches > 0]) else 0

    # Enunciation (clarity): zero crossing rate
    clarity_score = np.mean(librosa.feature.zero_crossing_rate(y=y))

    # Display feedback
    display(Audio(file_path))
    display_feedback(text, wpm, volume, filler_count, emotion_score, clarity_score)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown
import tempfile
from pydub import AudioSegment
import os

# Expanded accept types
uploader = widgets.FileUpload(
    accept='.wav,.mp3,.m4a,.aac,.ogg,.flac,.webm,.3gp',
    multiple=False
)

display(Markdown("###  Upload your speech recording (any common audio type)"))
display(uploader)

def on_upload_change(change):
    if uploader.value:
        for name, file_info in uploader.value.items():
            ext = os.path.splitext(name)[-1].lower()
            temp_original = tempfile.NamedTemporaryFile(delete=False, suffix=ext)
            temp_original.write(file_info['content'])
            temp_original.close()

            # Convert to WAV (required for most audio libraries)
            temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
            sound = AudioSegment.from_file(temp_original.name)
            sound.export(temp_wav.name, format="wav")
            print(f" Analyzing file: {name}")
            analyze_speech(temp_wav.name)

uploader.observe(on_upload_change, names='value')

In [ ]:
import gradio as gr
import os
import tempfile
import datetime
import matplotlib.pyplot as plt
import numpy as np
import whisper
import textstat
from pydub import AudioSegment
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize

# Initialize Whisper model
model = whisper.load_model("base")

# In-memory storage
users = {}
progress_data = {}
journal_entries = {}

# Filler words and emotion keywords
filler_words = {"um", "uh", "like", "you know", "so", "actually", "basically", "right"}
emotion_keywords = {"happy": "😊", "sad": "😢", "angry": "😠", "excited": "😃", "nervous": "😬"}

# Confidence quotes and coaching tips
confidence_quotes = [
    "Believe in your voice, it's worth hearing.",
    "Every word you speak builds confidence.",
    "You are more powerful than your hesitation."
]

coaching_tips = [
    "Practice pausing instead of using filler words.",
    "Speak slowly and enunciate each word clearly.",
    "Use vocal variety to convey emotion effectively.",
    "Record yourself and reflect on improvements."
]

# Authentication
def authenticate(username, password):
    if username in users:
        return users[username] == password
    else:
        users[username] = password
        progress_data[username] = []
        journal_entries[username] = []
        return True

# Analyze speech
def analyze_speech(audio_file, username):
    audio = whisper.load_audio(audio_file)
    audio = whisper.pad_or_trim(audio)
    mel = whisper.log_mel_spectrogram(audio).to(model.device)
    options = whisper.DecodingOptions(language="en", fp16=False)
    result = model.transcribe(audio_file)

    transcript = result['text']
    words = word_tokenize(transcript)
    wpm = len(words) / (len(audio) / 16000 / 60)
    filler_count = sum(word.lower() in filler_words for word in words)
    readability = textstat.flesch_reading_ease(transcript)

    detected_emotions = [word for word in words if word.lower() in emotion_keywords]
    emotion_feedback = ", ".join(set(detected_emotions)) if detected_emotions else "Neutral"

    # Save progress
    today = datetime.date.today().isoformat()
    progress_data[username].append({
        "date": today,
        "wpm": wpm,
        "filler": filler_count,
        "readability": readability
    })

    # Feedback string
    feedback = f"""
**Transcript:** {transcript}

**Words per Minute (WPM):** {wpm:.2f}
**Filler Words Used:** {filler_count}
**Readability Score:** {readability:.2f}
**Detected Emotions:** {emotion_feedback}

**Coaching Tips:**
- {np.random.choice(coaching_tips)}
- {np.random.choice(confidence_quotes)}
    """
    return feedback

# Plotting progress
def plot_progress(username):
    data = progress_data.get(username, [])
    if not data:
        return "No progress data yet.", None

    dates = [entry["date"] for entry in data]
    wpm = [entry["wpm"] for entry in data]
    filler = [entry["filler"] for entry in data]

    plt.figure(figsize=(10, 5))
    plt.plot(dates, wpm, marker='o', label='WPM')
    plt.plot(dates, filler, marker='x', label='Filler Words')
    plt.title('Progress Over Time')
    plt.xlabel('Date')
    plt.ylabel('Score')
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()

    tmpfile = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    plt.savefig(tmpfile.name)
    plt.close()
    return "Your progress chart is below:", tmpfile.name

# Journal saving
def save_journal(username, entry):
    today = datetime.date.today().isoformat()
    journal_entries[username].append((today, entry))
    return "Journal entry saved."

# Interface setup
with gr.Blocks() as demo:
    gr.Markdown("##  SpeakUP: Your Digital Speech Coach")

    username = gr.Textbox(label="Username")
    password = gr.Textbox(label="Password", type="password")
    auth_button = gr.Button("Login / Sign Up")
    auth_status = gr.Markdown("")

    with gr.Column(visible=False) as app:
        gr.Markdown("### Welcome to SpeakUP!")
        gr.Markdown(np.random.choice(confidence_quotes))

        audio_input = gr.Audio(label="Upload your speech (.wav preferred)", type="filepath")
        submit_audio = gr.Button("Analyze Speech")
        feedback_output = gr.Markdown("")

        gr.Markdown("###  Your Progress")
        show_chart = gr.Button("Show Progress Chart")
        chart_output = gr.Image()

        gr.Markdown("###  Daily Journal")
        journal_input = gr.Textbox(label="Write today's reflection...")
        journal_button = gr.Button("Save Journal Entry")
        journal_output = gr.Markdown("")

    def handle_login(user, pw):
        success = authenticate(user, pw)
        return (gr.update(visible=True), "✅ Logged in successfully!" if success else "❌ Login failed.")

    auth_button.click(handle_login, inputs=[username, password], outputs=[app, auth_status])

    def handle_audio(audio_file, user):
        return analyze_speech(audio_file, user)

    submit_audio.click(handle_audio, inputs=[audio_input, username], outputs=feedback_output)

    def handle_chart(user):
        return plot_progress(user)

    show_chart.click(handle_chart, inputs=[username], outputs=[chart_output, chart_output])

    journal_button.click(save_journal, inputs=[username, journal_input], outputs=journal_output)

demo.launch()


In [ ]:
!pip install gradio speechrecognition pydub librosa numpy --quiet
!pip install textstat
!pip install git+https://github.com/openai/whisper.git --quiet
!pip install nltk --quiet

In [ ]:


import gradio as gr
import speech_recognition as sr
from pydub import AudioSegment
from pydub.utils import mediainfo
import numpy as np
import io
import random
import tempfile
import matplotlib.pyplot as plt

# Coaching Tip Generators
def generate_coaching_tips(wpm, filler_count, avg_volume, emotion_score):
    tips = []
    if wpm < 110:
        tips.append("You're speaking a bit slowly. Try to increase your pace slightly to maintain audience engagement.")
    elif wpm > 160:
        tips.append("You're speaking quite fast. Practice pacing to ensure your message is clear and digestible.")
    else:
        tips.append("Your speaking speed is within a great range. Keep it up!")

    if filler_count > 10:
        tips.append("Consider reducing filler words like 'um' and 'uh'. Practice with pauses instead to sound more confident.")
    elif filler_count > 3:
        tips.append("You're doing okay with filler words, but there's still room for improvement.")
    else:
        tips.append("Great job avoiding filler words! Your speech sounds polished.")

    if avg_volume < -35:
        tips.append("Your volume is on the softer side. Speak up to make sure your audience hears you clearly.")
    elif avg_volume > -15:
        tips.append("You may be speaking too loudly. Try to keep a consistent and moderate volume.")
    else:
        tips.append("Your volume is balanced. Well done!")

    if emotion_score < 0.3:
        tips.append("Add more emotion and emphasis to your speech to engage the audience. Vary your pitch and stress key points.")
    elif emotion_score < 0.6:
        tips.append("You're showing some expression, but you can take it further.")
    else:
        tips.append("You're doing great with emotional expression! Keep connecting with your audience this way.")

    return "\n\n".join(tips)

def get_confidence_quote():
    quotes = [
        "Believe you can and you're halfway there. – Theodore Roosevelt",
        "Speak your truth—even if your voice shakes.",
        "Your voice matters. Use it to change the world.",
        "The way to gain confidence is to do what you are afraid to do.",
        "Practice makes progress. Every speech you give brings you one step closer to mastery."
    ]
    return random.choice(quotes)

def get_speech_goal():
    goals = [
        "Record one speech every day for the next 5 days.",
        "Practice with a timer and aim for 130–150 words per minute.",
        "Limit yourself to 2 filler words max in your next speech.",
        "Add a moment of dramatic pause in your next speech.",
        "Deliver your speech while standing in front of a mirror."
    ]
    return random.choice(goals)

# Main speech analysis logic
def analyze_speech(file):
    audio = AudioSegment.from_file(file)
    duration_seconds = len(audio) / 1000.0
    samples = np.array(audio.get_array_of_samples())
    avg_volume = 20 * np.log10(np.sqrt(np.mean(samples**2)) / 32768)

    recognizer = sr.Recognizer()
    with sr.AudioFile(file) as source:
        audio_data = recognizer.record(source)
    try:
        transcript = recognizer.recognize_google(audio_data)
    except sr.UnknownValueError:
        return "Couldn't understand the audio. Please try again.", None, None

    words = transcript.split()
    word_count = len(words)
    wpm = word_count / (duration_seconds / 60)

    filler_words = ['um', 'uh', 'like', 'you know']
    filler_count = sum(words.count(filler) for filler in filler_words)

    pitch = audio.low_pass_filter(3000).high_pass_filter(85)
    pitch_samples = np.array(pitch.get_array_of_samples())
    pitch_std = np.std(pitch_samples)
    emotion_score = min(pitch_std / 5000, 1.0)

    tips = generate_coaching_tips(wpm, filler_count, avg_volume, emotion_score)
    quote = get_confidence_quote()
    goal = get_speech_goal()

    summary = f"""
### Transcript:
{transcript}

### Analysis Summary:
- Words per minute (WPM): {wpm:.2f}
- Filler words detected: {filler_count}
- Average Volume: {avg_volume:.2f} dBFS
- Emotion Score: {emotion_score:.2f}

### Coaching Tips:
{tips}

### Confidence Quote:
"{quote}"

### Suggested Speech Goal:
{goal}
"""
    return summary

# Gradio Interface
gr.Interface(
    fn=analyze_speech,
    inputs=gr.Audio(type="filepath", label="Upload Your Speech (MP3 or WAV)"), # Remove source="upload"
    outputs=gr.Markdown(),
    title="SpeakUP: Digital Speech Coach",
    description="Upload a speech recording and receive instant feedback, coaching tips, confidence quotes, and speech goals!"
).launch()


In [ ]:
import gradio as gr
import os
import tempfile
import datetime
import matplotlib.pyplot as plt
import numpy as np
import whisper
import textstat
from pydub import AudioSegment
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize

# Initialize Whisper model
model = whisper.load_model("base")

# In-memory storage
users = {}
progress_data = {}
journal_entries = {}

# Filler words and emotion keywords
filler_words = {"um", "uh", "like", "you know", "so", "actually", "basically", "right"}
emotion_keywords = {"happy": "😊", "sad": "😢", "angry": "😠", "excited": "😃", "nervous": "😬"}

# Confidence quotes and coaching tips
confidence_quotes = [
    "Believe in your voice, it's worth hearing.",
    "Every word you speak builds confidence.",
    "You are more powerful than your hesitation."
]

coaching_tips = [
    "Practice pausing instead of using filler words.",
    "Speak slowly and enunciate each word clearly.",
    "Use vocal variety to convey emotion effectively.",
    "Record yourself and reflect on improvements."
]

# Authentication
def authenticate(username, password):
    if username in users:
        return users[username] == password
    else:
        users[username] = password
        progress_data[username] = []
        journal_entries[username] = []
        return True

# Analyze speech
def analyze_speech(audio_file, username):
    audio = whisper.load_audio(audio_file)
    audio = whisper.pad_or_trim(audio)
    mel = whisper.log_mel_spectrogram(audio).to(model.device)
    options = whisper.DecodingOptions(language="en", fp16=False)
    result = model.transcribe(audio_file)

    transcript = result['text']
    words = word_tokenize(transcript)
    wpm = len(words) / (len(audio) / 16000 / 60)
    filler_count = sum(word.lower() in filler_words for word in words)
    readability = textstat.flesch_reading_ease(transcript)

    detected_emotions = [word for word in words if word.lower() in emotion_keywords]
    emotion_feedback = ", ".join(set(detected_emotions)) if detected_emotions else "Neutral"

    # Save progress
    today = datetime.date.today().isoformat()
    progress_data[username].append({
        "date": today,
        "wpm": wpm,
        "filler": filler_count,
        "readability": readability
    })

    # Feedback string
    feedback = f"""
**Transcript:** {transcript}

**Words per Minute (WPM):** {wpm:.2f}
**Filler Words Used:** {filler_count}
**Readability Score:** {readability:.2f}
**Detected Emotions:** {emotion_feedback}

**Coaching Tips:**
- {np.random.choice(coaching_tips)}
- {np.random.choice(confidence_quotes)}
    """
    return feedback

# Plotting progress
def plot_progress(username):
    data = progress_data.get(username, [])
    if not data:
        return "No progress data yet.", None

    dates = [entry["date"] for entry in data]
    wpm = [entry["wpm"] for entry in data]
    filler = [entry["filler"] for entry in data]

    plt.figure(figsize=(10, 5))
    plt.plot(dates, wpm, marker='o', label='WPM')
    plt.plot(dates, filler, marker='x', label='Filler Words')
    plt.title('Progress Over Time')
    plt.xlabel('Date')
    plt.ylabel('Score')
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()

    tmpfile = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    plt.savefig(tmpfile.name)
    plt.close()
    return "Your progress chart is below:", tmpfile.name

# Journal saving
def save_journal(username, entry):
    today = datetime.date.today().isoformat()
    journal_entries[username].append((today, entry))
    return "Journal entry saved."

# Interface setup
with gr.Blocks() as demo:
    gr.Markdown("##  SpeakUP: Your Digital Speech Coach")

    username = gr.Textbox(label="Username")
    password = gr.Textbox(label="Password", type="password")
    auth_button = gr.Button("Login / Sign Up")
    auth_status = gr.Markdown("")

    with gr.Column(visible=False) as app:
        gr.Markdown("### Welcome to SpeakUP!")
        gr.Markdown(np.random.choice(confidence_quotes))

        audio_input = gr.Audio(label="Upload your speech (.wav preferred)", type="filepath")
        submit_audio = gr.Button("Analyze Speech")
        feedback_output = gr.Markdown("")

        gr.Markdown("###  Your Progress")
        show_chart = gr.Button("Show Progress Chart")
        chart_output = gr.Image()

        gr.Markdown("###  Daily Journal")
        journal_input = gr.Textbox(label="Write today's reflection...")
        journal_button = gr.Button("Save Journal Entry")
        journal_output = gr.Markdown("")

    def handle_login(user, pw):
        success = authenticate(user, pw)
        return (gr.update(visible=True), "✅ Logged in successfully!" if success else "❌ Login failed.")

    auth_button.click(handle_login, inputs=[username, password], outputs=[app, auth_status])

    def handle_audio(audio_file, user):
        return analyze_speech(audio_file, user)

    submit_audio.click(handle_audio, inputs=[audio_input, username], outputs=feedback_output)

    def handle_chart(user):
        return plot_progress(user)

    show_chart.click(handle_chart, inputs=[username], outputs=[chart_output, chart_output])

    journal_button.click(save_journal, inputs=[username, journal_input], outputs=journal_output)

demo.launch()